In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
import pandas as pd

df = pd.read_csv("creditcard.csv")

print(df.shape)
print(df.columns)
df.head()

(284807, 31)
Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='object')


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
label_column = "Class"   # change this to your real column name

X = df.drop(columns=[label_column])
y = df[label_column]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1,1).to(device)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32).reshape(-1,1).to(device)

In [6]:
print("X_train shape:", X_train.shape)

X_train shape: torch.Size([227845, 30])


In [7]:
class HybridModel(nn.Module):
    def __init__(self, num_features, d_model=32):
        super().__init__()

        # Transformer branch
        self.embedding = nn.Linear(1, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=64,
            dropout=0.3,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # MLP branch
        self.mlp = nn.Sequential(
            nn.Linear(num_features, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(d_model + 32, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        # Transformer branch
        xt = x.unsqueeze(-1)
        xt = self.embedding(xt)
        xt = self.transformer(xt)
        xt = xt.mean(dim=1)

        # MLP branch
        xm = self.mlp(x)

        # Combine
        x_combined = torch.cat([xt, xm], dim=1)

        return self.classifier(x_combined)

In [8]:
model = HybridModel(num_features=X_train.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [9]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [10]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.0257
Epoch 2, Loss: 0.0033
Epoch 3, Loss: 0.0031
Epoch 4, Loss: 0.0028
Epoch 5, Loss: 0.0026
Epoch 6, Loss: 0.0025
Epoch 7, Loss: 0.0025
Epoch 8, Loss: 0.0024
Epoch 9, Loss: 0.0023
Epoch 10, Loss: 0.0022
Epoch 11, Loss: 0.0021
Epoch 12, Loss: 0.0022
Epoch 13, Loss: 0.0021
Epoch 14, Loss: 0.0019
Epoch 15, Loss: 0.0020
Epoch 16, Loss: 0.0020
Epoch 17, Loss: 0.0020
Epoch 18, Loss: 0.0019
Epoch 19, Loss: 0.0020
Epoch 20, Loss: 0.0020


In [11]:
model.eval()
with torch.no_grad():
    logits_test = model(X_test)
    probs = torch.sigmoid(logits_test).cpu().numpy().flatten()

y_test_np = y_test.cpu().numpy().flatten()

preds = (probs >= 0.5).astype(int)

print("Hybrid Model Results")
print(classification_report(y_test_np, preds))
print("ROC-AUC:", roc_auc_score(y_test_np, probs))

Hybrid Model Results
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     56864
         1.0       0.79      0.83      0.81        98

    accuracy                           1.00     56962
   macro avg       0.90      0.91      0.90     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC: 0.9749520517267121


In [13]:
import pandas as pd
import numpy as np

# ==============================
# 1️⃣ Make sure you have:
# probs = model predicted probabilities
# y_test = true labels
# ==============================

# Example:
# probs = model.predict_proba(X_test)[:, 1]
# y_test = y_test.values  (if pandas)

# Convert to numpy safely
probs = np.array(probs).flatten()
y_true = y_test.detach().cpu().numpy().flatten()

# ==============================
# 2️⃣ Choose Threshold
# ==============================

threshold = 0.5   # you can change to your best threshold

pred_labels = (probs >= threshold).astype(int)

# Convert numeric labels to readable text
pred_text = np.where(pred_labels == 1, "Fraud", "Not Fraud")

# ==============================
# 3️⃣ Create Output DataFrame
# ==============================

output_df = pd.DataFrame({
    "Actual_Label": y_true,
    "Predicted_Label": pred_labels,
    "Prediction_Text": pred_text,
    "Fraud_Probability": probs
})

# ==============================
# 4️⃣ Save CSV
# ==============================

output_df.to_csv("fraud_predictions_output.csv", index=False)

print("CSV file saved successfully as fraud_predictions_output.csv")

CSV file saved successfully as fraud_predictions_output.csv
